# Lovelace Bubble Card YAML Playground

Use this notebook to experiment with the pop-up generator before copying the YAML back into Home Assistant.


1. Run the setup cell below.
2. Edit the entity lists for each category.
3. Click **Generate YAML** to refresh the configuration string.


In [ ]:
from __future__ import annotations

import ipywidgets as widgets
from IPython.display import display

from lovelace_yaml_generator import EXAMPLE_ENTITIES, generate_pop_up_yaml

CATEGORY_DETAILS = [
    ("Weather", "mdi:weather-partly-cloudy", "Outdoor and climate sensors."),
    ("Particulates", "mdi:chemical-weapon", "Fine particulate matter sensors."),
    ("Gases", "mdi:molecule", "Gaseous pollutant sensors."),
]

category_inputs = {}
accordion_children = []
for name, icon, hint in CATEGORY_DETAILS:
    textarea = widgets.Textarea(
        value="\n".join(EXAMPLE_ENTITIES.get(name, [])),
        placeholder="sensor.example_entity",
        layout=widgets.Layout(width="100%", height="120px"),
    )
    category_inputs[name] = textarea
    accordion_children.append(
        widgets.VBox(
            [
                widgets.HTML(
                    value=(
                        f"<p><b>Icon:</b> <code>{icon}</code><br/>"
                        f"<span style='color:#555;'>{hint}</span></p>"
                    )
                ),
                textarea,
            ]
        )
    )

accordion = widgets.Accordion(children=accordion_children)
for index, (name, _, _) in enumerate(CATEGORY_DETAILS):
    accordion.set_title(index, name)

status = widgets.HTML()
yaml_output = widgets.Textarea(
    value="",
    placeholder="Generated YAML will appear here",
    layout=widgets.Layout(width="100%", height="360px"),
)

def collect_entities():
    entities = {}
    for name, textarea in category_inputs.items():
        sensors = [
            line.strip()
            for line in textarea.value.splitlines()
            if line.strip()
        ]
        if sensors:
            entities[name] = sensors
    return entities

def update_yaml(*_):
    entities = collect_entities()
    if not entities:
        yaml_output.value = ""
        status.value = (
            "<span style='color:#666;'>Enter at least one entity to generate YAML.</span>"
        )
        return
    try:
        yaml_output.value = generate_pop_up_yaml(entities)
        status.value = (
            "<span style='color:green;'>Generated YAML for "
            f"{len(entities)} categories.</span>"
        )
    except Exception as exc:
        yaml_output.value = f"Error: {exc}"
        status.value = "<span style='color:red;'>Unable to generate YAML.</span>"

generate_button = widgets.Button(
    description="Generate YAML",
    icon="play",
    button_style="primary",
    layout=widgets.Layout(width="200px"),
)

generate_button.on_click(update_yaml)

update_yaml()

display(
    widgets.VBox(
        [
            widgets.HTML(
                value=(
                    "<p>Update the entity lists below and click <b>Generate YAML</b> "
                    "to render a bubble-card pop-up configuration. Temperature "
                    "sensors are inserted automatically after the first weather sensor.</p>"
                )
            ),
            accordion,
            widgets.HBox([generate_button, status]),
            yaml_output,
        ],
        layout=widgets.Layout(width="100%"),
    )
)
